# Bradford Bulls — Standardized Frame Extraction Pipeline
## Production v1.0 — ALL team members use this notebook

### RULES
1. **DO NOT** modify any cell except **Cell 1 (Your Config)**
2. **DO NOT** write your own extraction script
3. **DO NOT** manually delete or filter frames after extraction
4. If you encounter issues, report to Lead — do not improvise

### Video Naming Convention (on Google Drive)
Upload your video to `Bradford_Bulls/videos/` with this exact format:
```
M{number}_{kit_color}_{resolution}.mp4
```

| Example | Meaning |
|---------|---------|
| `M01_white_1080p.mp4` | Match 1, home white kit, 1080p |
| `M02_black_1080p.mp4` | Match 2, away black kit, 1080p |
| `M03_white_720p.mp4`  | Match 3, home white kit, 720p |
| `M06_white_1080p_highlight.mp4` | Match 6, highlight reel |

### Pipeline
```
Video (Google Drive)
  → L1: Temporal sampling (1 FPS)
  → L2: Scene change detection (pHash + SSIM)
  → L3: Player presence + size filter (YOLO)
  → L4: Quality scoring + diversity selection
  → Save to Google Drive (frames + metadata CSV)
  → Ready for Roboflow upload
```

### Output Structure
```
Bradford_Bulls/
├── videos/
│   └── M01_white_1080p.mp4
├── frames/
│   └── M01_white_1080p/
│       ├── M01_white_1080p_000045_00m45s.jpg
│       ├── M01_white_1080p_000132_02m12s.jpg
│       └── ...
├── metadata/
│   └── M01_white_1080p_index.csv
└── reports/
    └── M01_white_1080p_report.txt
```

## 0. Install Dependencies
Run this cell first, then **Restart Runtime** before continuing.

In [ ]:
!pip install -q ultralytics>=8.3.0 imagehash scikit-image pillow opencv-python-headless roboflow
print("\nDone. Please RESTART RUNTIME now (Runtime > Restart runtime), then continue from Cell 1.")

## 1. YOUR CONFIG (ONLY cell you edit)

**Instructions:**
1. Set `MEMBER_NAME` to your name (no spaces, lowercase)
2. Set `VIDEO_FILENAME` to **exactly** the filename of your video on Google Drive
3. Set `TEST_MODE = True` for first run to verify everything works, then set `False` for full run

In [ ]:
# ============================================================
# YOUR CONFIG — Edit ONLY these 3 lines
# ============================================================

MEMBER_NAME = "your_name"                    # e.g. "hoa", "minh", "tuan"
VIDEO_FILENAME = "M01_white_1080p.mp4"       # Exact filename on Google Drive
TEST_MODE = True                             # True = test with 2000 frames, False = full video

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================

## 2. Setup Environment & Mount Google Drive

In [ ]:
import torch
import cv2
import os
import json
import csv
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imagehash
from pathlib import Path
from collections import Counter
from datetime import datetime
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from skimage.metrics import structural_similarity as compare_ssim

# --- GPU Check ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. This will be slow.")

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Directory Structure ---
DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")
VIDEOS_DIR = DRIVE_ROOT / "videos"
FRAMES_DIR = DRIVE_ROOT / "frames"
METADATA_DIR = DRIVE_ROOT / "metadata"
REPORTS_DIR = DRIVE_ROOT / "reports"

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Derive match_id from video filename ---
# Expected format: M01_white_1080p.mp4 or M01_white_1080p_highlight.mp4
MATCH_ID = Path(VIDEO_FILENAME).stem  # e.g. "M01_white_1080p"

# Output directories for THIS video
FRAMES_OUTPUT_DIR = FRAMES_DIR / MATCH_ID
FRAMES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Validate video exists ---
VIDEO_PATH = VIDEOS_DIR / VIDEO_FILENAME
if not VIDEO_PATH.exists():
    available = list(VIDEOS_DIR.glob("*.mp4"))
    print(f"ERROR: '{VIDEO_FILENAME}' not found in {VIDEOS_DIR}")
    print(f"\nAvailable videos:")
    for v in available:
        size_mb = v.stat().st_size / 1e6
        print(f"  - {v.name} ({size_mb:.0f} MB)")
    if not available:
        print("  (none — please upload your video first)")
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

print(f"\nVideo found: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size/1e6:.0f} MB)")
print(f"Member: {MEMBER_NAME}")
print(f"Match ID: {MATCH_ID}")
print(f"Output: {FRAMES_OUTPUT_DIR}")

## 3. Load Video & Display Info

In [ ]:
# --- Read video metadata ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise ValueError(f"Cannot open video: {VIDEO_PATH}")

FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION_SEC = TOTAL_FRAMES / FPS
cap.release()

# --- Standardized extraction parameters (LOCKED) ---
TARGET_FPS = 1                      # 1 frame per second
PHASH_THRESHOLD = 10                # Perceptual hash dedup threshold
SSIM_THRESHOLD = 0.92               # Scene similarity threshold
PERSON_CONFIDENCE = 0.5             # YOLO person detection confidence
MIN_PERSONS = 1                     # Minimum persons in frame
MIN_MAX_PERSON_AREA_RATIO = 0.03    # Reject wide shots (person < 3% of frame)
MIN_SHARPNESS = 0.25                # Sharpness threshold (0-1)
ENABLE_PITCH_GREEN_FILTER = True    # Filter non-pitch frames
PITCH_ROI_Y_START = 0.40            # Analyze bottom 60% of frame
MIN_PITCH_GREEN_RATIO = 0.06        # Min grass-green pixel ratio
DEDUP_HASH_THRESH = 6               # Hash distance for final dedup
TIME_BUCKET_SEC = 10                # Diversity: max frames per N-second window
PERSON_DETECTION_MODEL = "yolov8m.pt"  # Model for person detection (medium for accuracy)
JPEG_QUALITY = 95                   # Output quality
INCLUDE_MOTION_BLUR_FRAMES = True   # Keep some blurry frames for training diversity
MOTION_BLUR_SHARPNESS_MIN = 0.12    # Floor for motion blur frames (below this = unusable)
MOTION_BLUR_QUOTA = 0.15            # Max 15% of selected frames can be motion-blurred

# Frame interval based on video FPS
FRAME_INTERVAL = max(1, int(FPS / TARGET_FPS))

# Test mode limits
if TEST_MODE:
    MAX_SCAN_FRAMES = 2000
    TARGET_SELECTED = 30
    print(f"TEST MODE: scanning {MAX_SCAN_FRAMES} frames, selecting {TARGET_SELECTED}")
else:
    MAX_SCAN_FRAMES = TOTAL_FRAMES
    TARGET_SELECTED = None  # Select all that pass filters
    print(f"PRODUCTION MODE: scanning all {TOTAL_FRAMES:,} frames")

print(f"\n{'='*60}")
print(f"  VIDEO INFO")
print(f"{'='*60}")
print(f"  File:        {VIDEO_PATH.name}")
print(f"  Duration:    {DURATION_SEC/60:.1f} min ({DURATION_SEC:.0f} sec)")
print(f"  Resolution:  {W}x{H}")
print(f"  FPS:         {FPS:.0f}")
print(f"  Total:       {TOTAL_FRAMES:,} frames")
print(f"  Sample rate: 1 frame every {FRAME_INTERVAL} frames ({TARGET_FPS} FPS)")
print(f"  Estimated L1 frames: ~{TOTAL_FRAMES // FRAME_INTERVAL:,}")
print(f"{'='*60}")

## 4. Helper Functions (Locked)

In [ ]:
def fmt_timestamp(sec):
    """Convert seconds to HH:MM:SS string for filenames."""
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"


def compute_sharpness(frame):
    """Multi-metric sharpness score (0-1). Higher = sharper."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Laplacian variance
    lap = cv2.Laplacian(gray, cv2.CV_64F).var()
    # Tenengrad (Sobel gradient magnitude)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    ten = np.mean(gx**2 + gy**2)
    # Normalized variance
    mean_val = gray.mean()
    nvar = gray.var() / (mean_val + 1e-6)
    # Weighted combination, capped at 1.0
    score = (
        min(lap / 300, 1.0) * 0.40 +
        min(ten / 5000, 1.0) * 0.35 +
        min(nvar / 50, 1.0) * 0.25
    )
    return round(score, 4)


def compute_phash(frame_bgr):
    """Compute perceptual hash of a frame."""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return imagehash.phash(Image.fromarray(frame_rgb))


def compute_ssim(frame1, frame2):
    """Compute structural similarity between two frames."""
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    if gray1.shape != gray2.shape:
        gray2 = cv2.resize(gray2, (gray1.shape[1], gray1.shape[0]))
    return float(compare_ssim(gray1, gray2))


def compute_pitch_green_ratio(frame_bgr, roi_y_start=0.40):
    """Fraction of grass-green pixels in bottom ROI (0-1)."""
    h, w = frame_bgr.shape[:2]
    y0 = int(max(0, min(h - 1, roi_y_start * h)))
    roi = frame_bgr[y0:h, 0:w]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower = np.array([35, 40, 40])
    upper = np.array([95, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    return float(mask.mean() / 255.0)


def detect_persons_batch(model, frames, device, confidence=0.5):
    """Batch detect persons, return list of detection dicts per frame."""
    results = model.predict(
        frames, classes=[0], conf=confidence,
        device=device, verbose=False, batch=len(frames)
    )
    all_detections = []
    for res in results:
        detections = []
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            confs = res.boxes.conf.cpu().numpy()
            for i in range(len(xyxy)):
                x1, y1, x2, y2 = xyxy[i]
                detections.append({
                    "bbox": [float(x1), float(y1), float(x2), float(y2)],
                    "conf": float(confs[i]),
                    "area": float((x2 - x1) * (y2 - y1)),
                })
        all_detections.append(detections)
    return all_detections


print("Helper functions loaded.")

## 5. Frame Extraction — 4-Layer Pipeline

This is the main extraction loop:
- **L1 (Temporal):** Sample 1 frame/second from original video
- **L2 (Scene Change):** Skip frames too similar to previous (pHash + SSIM)
- **L3 (Player Presence):** Keep only frames with visible players, reject wide shots and non-pitch
- **L4 (Quality Score):** Score each frame by sharpness + player size + diversity

**Important:** This intentionally keeps some motion-blurred frames (controlled by `MOTION_BLUR_QUOTA`).
Model needs to learn to detect logos in imperfect conditions.

In [ ]:
# --- Load YOLO model ---
print(f"Loading {PERSON_DETECTION_MODEL}...")
yolo_model = YOLO(PERSON_DETECTION_MODEL)
print(f"Model loaded on {DEVICE}")

# --- Main extraction loop ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
frame_area = W * H

candidates = []       # Frames that pass L1-L3
stats = {
    "total_scanned": 0,
    "l1_sampled": 0,
    "l2_skipped_phash": 0,
    "l2_skipped_ssim": 0,
    "l3_skipped_no_person": 0,
    "l3_skipped_wide_shot": 0,
    "l3_skipped_no_pitch": 0,
    "l3_skipped_too_blurry": 0,
    "l3_passed": 0,
}

prev_phash = None
prev_frame = None

# Batch processing buffers
batch_frames = []
batch_frame_nums = []
batch_phashes = []
BATCH_SIZE = 16

pbar = tqdm(total=min(MAX_SCAN_FRAMES, TOTAL_FRAMES), desc="Scanning", unit="fr")

def process_batch(batch_frames, batch_frame_nums, batch_phashes):
    """Process a batch of frames through L3 (person detection + quality)."""
    if not batch_frames:
        return

    # Batch person detection
    all_detections = detect_persons_batch(
        yolo_model, batch_frames, DEVICE, PERSON_CONFIDENCE
    )

    for frame, frame_num, phash, detections in zip(
        batch_frames, batch_frame_nums, batch_phashes, all_detections
    ):
        # L3a: Minimum persons
        if len(detections) < MIN_PERSONS:
            stats["l3_skipped_no_person"] += 1
            continue

        # L3b: Wide shot rejection
        max_person_area_ratio = max(d["area"] / frame_area for d in detections)
        if max_person_area_ratio < MIN_MAX_PERSON_AREA_RATIO:
            stats["l3_skipped_wide_shot"] += 1
            continue

        # L3c: Pitch green filter
        pitch_green = compute_pitch_green_ratio(frame, PITCH_ROI_Y_START)
        if ENABLE_PITCH_GREEN_FILTER and pitch_green < MIN_PITCH_GREEN_RATIO:
            stats["l3_skipped_no_pitch"] += 1
            continue

        # L4: Quality scoring
        sharpness = compute_sharpness(frame)

        # Sharpness gate with motion blur allowance
        is_motion_blur = sharpness < MIN_SHARPNESS
        if is_motion_blur:
            if not INCLUDE_MOTION_BLUR_FRAMES or sharpness < MOTION_BLUR_SHARPNESS_MIN:
                stats["l3_skipped_too_blurry"] += 1
                continue

        # Compute quality score
        total_player_area = sum(d["area"] for d in detections)
        player_coverage = min(total_player_area / frame_area, 1.0)

        score = (
            sharpness * 0.40 +
            min(len(detections) / 6.0, 1.0) * 0.30 +
            player_coverage * 0.30
        )

        timestamp_sec = frame_num / FPS
        candidates.append({
            "frame_num": frame_num,
            "timestamp_sec": round(timestamp_sec, 2),
            "timestamp_hms": fmt_timestamp(timestamp_sec),
            "n_persons": len(detections),
            "sharpness": sharpness,
            "is_motion_blur": is_motion_blur,
            "player_coverage": round(player_coverage, 4),
            "max_person_area_ratio": round(max_person_area_ratio, 4),
            "pitch_green_ratio": round(pitch_green, 4),
            "score": round(score, 4),
            "phash": str(phash),
        })
        stats["l3_passed"] += 1


frame_num = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_num >= MAX_SCAN_FRAMES:
        break

    pbar.update(1)
    stats["total_scanned"] += 1

    # L1: Temporal sampling
    if frame_num % FRAME_INTERVAL != 0:
        frame_num += 1
        continue

    stats["l1_sampled"] += 1

    # L2: Scene change detection (pHash + SSIM)
    current_phash = compute_phash(frame)

    if prev_phash is not None:
        hash_diff = current_phash - prev_phash
        if hash_diff < PHASH_THRESHOLD:
            # Hashes similar — confirm with SSIM
            similarity = compute_ssim(prev_frame, frame)
            if similarity > SSIM_THRESHOLD:
                stats["l2_skipped_ssim"] += 1
                frame_num += 1
                continue

    prev_phash = current_phash
    prev_frame = frame.copy()

    # Add to batch for L3 processing
    batch_frames.append(frame.copy())
    batch_frame_nums.append(frame_num)
    batch_phashes.append(current_phash)

    if len(batch_frames) >= BATCH_SIZE:
        process_batch(batch_frames, batch_frame_nums, batch_phashes)
        batch_frames, batch_frame_nums, batch_phashes = [], [], []

    frame_num += 1

# Process remaining batch
process_batch(batch_frames, batch_frame_nums, batch_phashes)

pbar.close()
cap.release()

# Sort by score (best first)
candidates.sort(key=lambda x: x["score"], reverse=True)

# --- Print stats ---
print(f"\n{'='*60}")
print(f"  EXTRACTION STATS — {MATCH_ID}")
print(f"{'='*60}")
print(f"  Total scanned:        {stats['total_scanned']:,}")
print(f"  L1 sampled ({TARGET_FPS} FPS):  {stats['l1_sampled']:,}")
print(f"  L2 skipped (similar): {stats['l2_skipped_ssim']:,}")
after_l2 = stats['l1_sampled'] - stats['l2_skipped_ssim']
print(f"  After L2:             {after_l2:,}")
print(f"  L3 rejected:")
print(f"    No person:          {stats['l3_skipped_no_person']:,}")
print(f"    Wide shot:          {stats['l3_skipped_wide_shot']:,}")
print(f"    No pitch:           {stats['l3_skipped_no_pitch']:,}")
print(f"    Too blurry:         {stats['l3_skipped_too_blurry']:,}")
print(f"  CANDIDATES:           {len(candidates):,}")
if candidates:
    blur_count = sum(1 for c in candidates if c["is_motion_blur"])
    print(f"    Sharp frames:       {len(candidates) - blur_count:,}")
    print(f"    Motion blur frames: {blur_count:,} ({blur_count/len(candidates)*100:.1f}%)")
    print(f"    Score range:        {candidates[-1]['score']:.3f} — {candidates[0]['score']:.3f}")
print(f"{'='*60}")

## 6. De-duplicate & Select Final Frames

Selects the best frames while ensuring:
- No near-duplicates (pHash distance)
- Time diversity (max N frames per 10-second window)
- Motion blur quota (max 15% blurry frames — needed for training diversity)

In **production mode**, all qualifying frames are kept (no target limit).
In **test mode**, only top N are selected.

In [ ]:
# --- De-duplicate and select ---
selected = []
selected_hashes = []
time_bucket_counts = Counter()
motion_blur_count = 0

# Compute max frames per time bucket for diversity
if TARGET_SELECTED:
    num_buckets = max(1, int(DURATION_SEC / TIME_BUCKET_SEC))
    MAX_PER_BUCKET = max(2, TARGET_SELECTED // num_buckets + 2)
    max_blur = int(np.ceil(TARGET_SELECTED * MOTION_BLUR_QUOTA))
else:
    MAX_PER_BUCKET = 5  # production: allow up to 5 per 10s window
    max_blur = int(np.ceil(len(candidates) * MOTION_BLUR_QUOTA))

print(f"Selecting from {len(candidates)} candidates...")
print(f"  Max per {TIME_BUCKET_SEC}s window: {MAX_PER_BUCKET}")
print(f"  Motion blur quota: max {max_blur} frames ({MOTION_BLUR_QUOTA:.0%})")
if TARGET_SELECTED:
    print(f"  Target: {TARGET_SELECTED} frames (test mode)")
else:
    print(f"  Target: all qualifying frames (production mode)")

cap = cv2.VideoCapture(str(VIDEO_PATH))

for meta in tqdm(candidates, desc="Selecting"):
    # Time diversity check
    bucket = int(meta["timestamp_sec"] // TIME_BUCKET_SEC)
    if time_bucket_counts[bucket] >= MAX_PER_BUCKET:
        continue

    # Motion blur quota
    if meta["is_motion_blur"]:
        if motion_blur_count >= max_blur:
            continue

    # Read frame for hash dedup
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret:
        continue

    frame_hash = compute_phash(frame)

    # Dedup check
    if any(frame_hash - h < DEDUP_HASH_THRESH for h in selected_hashes):
        continue

    # Accept frame
    selected.append(meta)
    selected_hashes.append(frame_hash)
    time_bucket_counts[bucket] += 1
    if meta["is_motion_blur"]:
        motion_blur_count += 1

    # Check target limit (test mode only)
    if TARGET_SELECTED and len(selected) >= TARGET_SELECTED:
        break

cap.release()

# Sort by timestamp for sequential output
selected.sort(key=lambda x: x["timestamp_sec"])

# --- Summary ---
blur_selected = sum(1 for s in selected if s["is_motion_blur"])
sharp_selected = len(selected) - blur_selected
print(f"\n{'='*60}")
print(f"  SELECTION RESULTS — {MATCH_ID}")
print(f"{'='*60}")
print(f"  Total selected:    {len(selected)} frames")
print(f"    Sharp:           {sharp_selected} ({sharp_selected/max(len(selected),1)*100:.0f}%)")
print(f"    Motion blur:     {blur_selected} ({blur_selected/max(len(selected),1)*100:.0f}%)")
if selected:
    scores = [s["score"] for s in selected]
    players = [s["n_persons"] for s in selected]
    print(f"  Score range:       {min(scores):.3f} — {max(scores):.3f}")
    print(f"  Players/frame:     {min(players)} — {max(players)} (avg {np.mean(players):.1f})")
    time_spread = selected[-1]["timestamp_sec"] - selected[0]["timestamp_sec"]
    print(f"  Time span:         {fmt_timestamp(selected[0]['timestamp_sec'])} — {fmt_timestamp(selected[-1]['timestamp_sec'])} ({time_spread/60:.1f} min)")
print(f"{'='*60}")

## 7. Save Frames & Metadata to Google Drive

Saves frames with standardized naming:
```
{MATCH_ID}_{frame_num:06d}_{timestamp}.jpg
```

Also saves a detailed CSV index for the Lead to review quality.

In [ ]:
# --- Save selected frames ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []
saved_count = 0

for idx, meta in enumerate(tqdm(selected, desc="Saving frames")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret:
        continue

    # Standardized filename
    fname = f"{MATCH_ID}_{meta['frame_num']:06d}_{meta['timestamp_hms']}.jpg"
    fpath = FRAMES_OUTPUT_DIR / fname
    cv2.imwrite(str(fpath), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    saved_count += 1

    rows.append({
        "id": idx + 1,
        "filename": fname,
        "frame_num": meta["frame_num"],
        "timestamp_sec": meta["timestamp_sec"],
        "timestamp_hms": meta["timestamp_hms"],
        "n_persons": meta["n_persons"],
        "sharpness": meta["sharpness"],
        "is_motion_blur": meta["is_motion_blur"],
        "player_coverage": meta["player_coverage"],
        "max_person_area_ratio": meta["max_person_area_ratio"],
        "pitch_green_ratio": meta["pitch_green_ratio"],
        "score": meta["score"],
        "match_id": MATCH_ID,
        "video_file": VIDEO_FILENAME,
        "resolution": f"{W}x{H}",
        "member": MEMBER_NAME,
        "extracted_at": datetime.now().isoformat(),
    })

cap.release()

# --- Save metadata CSV ---
df = pd.DataFrame(rows)
csv_path = METADATA_DIR / f"{MATCH_ID}_index.csv"
df.to_csv(csv_path, index=False)

print(f"\nSaved {saved_count} frames to: {FRAMES_OUTPUT_DIR}")
print(f"Metadata CSV: {csv_path}")

# --- Save extraction report ---
report_lines = [
    f"FRAME EXTRACTION REPORT",
    f"{'='*50}",
    f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"Member:         {MEMBER_NAME}",
    f"Video:          {VIDEO_FILENAME}",
    f"Match ID:       {MATCH_ID}",
    f"Resolution:     {W}x{H}",
    f"Duration:       {DURATION_SEC/60:.1f} min",
    f"FPS:            {FPS:.0f}",
    f"Total frames:   {TOTAL_FRAMES:,}",
    f"",
    f"PIPELINE STATS",
    f"{'='*50}",
    f"L1 sampled:     {stats['l1_sampled']:,}",
    f"L2 dedup:       -{stats['l2_skipped_ssim']:,}",
    f"L3 no person:   -{stats['l3_skipped_no_person']:,}",
    f"L3 wide shot:   -{stats['l3_skipped_wide_shot']:,}",
    f"L3 no pitch:    -{stats['l3_skipped_no_pitch']:,}",
    f"L3 too blurry:  -{stats['l3_skipped_too_blurry']:,}",
    f"Candidates:     {len(candidates):,}",
    f"Final selected: {saved_count}",
    f"  Sharp:        {sharp_selected}",
    f"  Motion blur:  {blur_selected}",
    f"",
    f"PARAMETERS (locked)",
    f"{'='*50}",
    f"TARGET_FPS:              {TARGET_FPS}",
    f"PHASH_THRESHOLD:         {PHASH_THRESHOLD}",
    f"SSIM_THRESHOLD:          {SSIM_THRESHOLD}",
    f"PERSON_CONFIDENCE:       {PERSON_CONFIDENCE}",
    f"MIN_PERSONS:             {MIN_PERSONS}",
    f"MIN_MAX_PERSON_AREA:     {MIN_MAX_PERSON_AREA_RATIO}",
    f"MIN_SHARPNESS:           {MIN_SHARPNESS}",
    f"MOTION_BLUR_MIN:         {MOTION_BLUR_SHARPNESS_MIN}",
    f"MOTION_BLUR_QUOTA:       {MOTION_BLUR_QUOTA}",
    f"PITCH_GREEN_FILTER:      {ENABLE_PITCH_GREEN_FILTER}",
    f"MIN_PITCH_GREEN_RATIO:   {MIN_PITCH_GREEN_RATIO}",
    f"DEDUP_HASH_THRESH:       {DEDUP_HASH_THRESH}",
    f"PERSON_DETECTION_MODEL:  {PERSON_DETECTION_MODEL}",
]

report_path = REPORTS_DIR / f"{MATCH_ID}_report.txt"
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))

print(f"Report: {report_path}")

## 8. Preview Extracted Frames

Shows a grid of sample frames. Check:
- Are players clearly visible?
- Is there variety in camera angles and timestamps?
- Are motion blur frames reasonable (logo partially visible)?

In [ ]:
# --- Preview: Top scored frames ---
frames_on_disk = sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg"))

if not frames_on_disk:
    print("No frames saved yet.")
else:
    # Show top 12 by score and bottom 4 (motion blur examples)
    df_preview = df.sort_values("score", ascending=False)

    # Top 12 best
    top_n = min(12, len(df_preview))
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    axes = axes.flatten()

    for i, (_, row) in enumerate(df_preview.head(top_n).iterrows()):
        fpath = FRAMES_OUTPUT_DIR / row["filename"]
        if fpath.exists():
            img = cv2.imread(str(fpath))
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            blur_tag = " [BLUR]" if row["is_motion_blur"] else ""
            axes[i].set_title(
                f"{row['timestamp_hms']} | score={row['score']:.2f} | "
                f"sharp={row['sharpness']:.2f} | players={row['n_persons']}{blur_tag}",
                fontsize=8
            )
        axes[i].axis("off")
    for j in range(top_n, len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"TOP {top_n} frames — {MATCH_ID}", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Show motion blur frames if any
    blur_frames = df_preview[df_preview["is_motion_blur"] == True]
    if len(blur_frames) > 0:
        n_blur_show = min(4, len(blur_frames))
        fig, axes = plt.subplots(1, n_blur_show, figsize=(5*n_blur_show, 5))
        if n_blur_show == 1:
            axes = [axes]
        for i, (_, row) in enumerate(blur_frames.head(n_blur_show).iterrows()):
            fpath = FRAMES_OUTPUT_DIR / row["filename"]
            if fpath.exists():
                img = cv2.imread(str(fpath))
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                axes[i].set_title(
                    f"BLUR | {row['timestamp_hms']} | sharp={row['sharpness']:.2f}",
                    fontsize=9
                )
            axes[i].axis("off")
        plt.suptitle(f"Motion Blur Samples — {MATCH_ID} (kept for training diversity)", fontsize=12)
        plt.tight_layout()
        plt.show()

    print(f"\nTotal frames on Drive: {len(frames_on_disk)}")
    print(f"Frames directory: {FRAMES_OUTPUT_DIR}")

## 9. Quality Distribution Analysis

Visualize the distribution of quality metrics across all extracted frames.
This helps the Lead verify consistency across different team members' videos.

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Sharpness distribution
    axes[0,0].hist(df["sharpness"], bins=30, color="steelblue", edgecolor="white")
    axes[0,0].axvline(MIN_SHARPNESS, color="red", linestyle="--", label=f"Threshold={MIN_SHARPNESS}")
    axes[0,0].set_title("Sharpness Distribution")
    axes[0,0].set_xlabel("Sharpness Score")
    axes[0,0].legend()

    # 2. Score distribution
    axes[0,1].hist(df["score"], bins=30, color="forestgreen", edgecolor="white")
    axes[0,1].set_title("Overall Quality Score")
    axes[0,1].set_xlabel("Score")

    # 3. Players per frame
    axes[0,2].hist(df["n_persons"], bins=range(0, df["n_persons"].max()+2),
                   color="coral", edgecolor="white", align="left")
    axes[0,2].set_title("Players per Frame")
    axes[0,2].set_xlabel("Number of Detected Persons")

    # 4. Max person area ratio
    axes[1,0].hist(df["max_person_area_ratio"], bins=30, color="mediumpurple", edgecolor="white")
    axes[1,0].axvline(MIN_MAX_PERSON_AREA_RATIO, color="red", linestyle="--", label=f"Min={MIN_MAX_PERSON_AREA_RATIO}")
    axes[1,0].set_title("Largest Person Size (% of frame)")
    axes[1,0].set_xlabel("Area Ratio")
    axes[1,0].legend()

    # 5. Temporal distribution
    axes[1,1].hist(df["timestamp_sec"]/60, bins=30, color="goldenrod", edgecolor="white")
    axes[1,1].set_title("Temporal Distribution")
    axes[1,1].set_xlabel("Time (minutes)")
    axes[1,1].set_ylabel("Frames")

    # 6. Pitch green ratio
    axes[1,2].hist(df["pitch_green_ratio"], bins=30, color="green", edgecolor="white")
    axes[1,2].axvline(MIN_PITCH_GREEN_RATIO, color="red", linestyle="--", label=f"Min={MIN_PITCH_GREEN_RATIO}")
    axes[1,2].set_title("Pitch Green Ratio")
    axes[1,2].set_xlabel("Green Pixel Ratio")
    axes[1,2].legend()

    plt.suptitle(f"Quality Metrics — {MATCH_ID} ({len(df)} frames)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Summary table
    print(f"\n{'='*50}")
    print(f"  QUALITY SUMMARY — {MATCH_ID}")
    print(f"{'='*50}")
    print(df[["sharpness", "score", "n_persons", "max_person_area_ratio", "pitch_green_ratio"]].describe().round(3).to_string())
else:
    print("No data to analyze.")

## 10. Upload to Roboflow (Optional)

Upload extracted frames to Roboflow for annotation.
Only run this if Lead has confirmed the frames are ready.

In [ ]:
# --- Roboflow Upload ---
# ONLY run this cell after Lead approves your extracted frames

ROBOFLOW_API_KEY = "YOUR_API_KEY"   # Get from Lead
ROBOFLOW_PROJECT = "kit-sponsor-logos"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()

try:
    project = workspace.project(ROBOFLOW_PROJECT)
    print(f"Connected to project: {project.name}")
except Exception:
    print(f"Project '{ROBOFLOW_PROJECT}' not found. Ask Lead to create it first.")
    raise

frames_to_upload = sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg"))
print(f"Uploading {len(frames_to_upload)} frames from {MATCH_ID}...")

success, errors = 0, 0
for fp in tqdm(frames_to_upload, desc="Uploading"):
    try:
        project.upload(image_path=str(fp), split="train", tag=[MATCH_ID, MEMBER_NAME])
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  Error: {e}")

print(f"\nUpload complete: {success} success, {errors} errors")
print(f"Annotate at: https://app.roboflow.com/{workspace.url}/{ROBOFLOW_PROJECT}/annotate")

## 11. Checklist Before Submitting

Before notifying Lead that your extraction is complete, verify:

- [ ] `TEST_MODE = False` and full video was processed
- [ ] Frames are saved to `Bradford_Bulls/frames/{MATCH_ID}/`
- [ ] Metadata CSV exists at `Bradford_Bulls/metadata/{MATCH_ID}_index.csv`
- [ ] Report exists at `Bradford_Bulls/reports/{MATCH_ID}_report.txt`
- [ ] Preview shows reasonable frames (players visible, variety of timestamps)
- [ ] No errors during extraction

**Report to Lead:**
1. Your `MATCH_ID`
2. Number of frames extracted
3. Any issues or unusual observations (e.g., "many frames rejected by pitch filter" or "video quality is low")